# Session 1 — Solutions

This notebook contains worked solutions for Session 1.

## Loading events and counting (Ex 1.1)


In [ ]:
from coffea.nanoevents import NanoEventsFactory, NanoAODSchema


def load_events(filepath):
    """Load one NanoAOD ROOT file and return a NanoEvents object.

    - **Why NanoEvents?** It maps flat NanoAOD branches like `Jet_pt`, `MET_pt` into
      convenient collections like `events.Jet.pt`, `events.MET.pt` using Awkward Arrays.
    """
    return NanoEventsFactory.from_root(filepath, schemaclass=NanoAODSchema).events()


# Load events and print basic information.
# We add the project root to sys.path so `config/` can be imported in SWAN.
import sys
sys.path.append("..")
from config.datasets_2017 import get_one_file_per_group_from_yaml

try:
    files = get_one_file_per_group_from_yaml()

    # We load one *data* file (MET run) and one *MC background* file.
    # This is enough for basic object/branch exploration and plotting.
    events_data = load_events(files["data"][0])
    events = load_events(files["background"][0])

    # Run/Lumi/Event numbers uniquely identify events in CMS data.
    print("Total events:", len(events_data))
    print("run:", events_data.run[:5])
    print("luminosityBlock:", events_data.luminosityBlock[:5])
    print("event:", events_data.event[:5])

except Exception as e:
    print("Could not load events.")
    print(e)

In [ ]:
import matplotlib.pyplot as plt
import awkward as ak
import mplhep as hep

hep.style.use("CMS")

## Inspect branches (Ex 1.2)


In [ ]:
print("Jet fields:", events_data.Jet.fields)

## Basic plotting (no specific exercise)

As a first example of plotting with matplotlib, we can make a simple MET distribution using the loaded `events` object.

In [ ]:
# Example: simple MET histogram
# MET (pTmiss) is defined in the transverse plane and acts as a proxy for invisible
# particles that escape the detector (neutrinos / DM).
met = events.MET.pt

# Convert Awkward Array → NumPy for matplotlib.
plt.figure()
plt.hist(ak.to_numpy(met), bins=50, range=(0, 400), edgecolor="black", alpha=0.7)
plt.xlabel("MET [GeV]")
plt.ylabel("Events")
plt.title("Missing transverse energy")
plt.show()

## Jet pT plot (Ex 1.3)

Fill a histogram of jet transverse momentum for all jets in the sample.

In [ ]:
# Solution: jet pT histogram
# `events.Jet.pt` is *jagged* (variable number of jets per event).
# We flatten it to get one pT value per jet across all events.
jpt = ak.flatten(events.Jet.pt)

plt.figure()
plt.hist(ak.to_numpy(jpt), bins=50, range=(0, 500), edgecolor="black", alpha=0.7)
plt.xlabel("Jet p$_T$ [GeV]")
plt.ylabel("Jets")
plt.title("Jet transverse momentum")
plt.show()

## Jet multiplicity (Ex 1.4)

Plot the distribution of the number of jets per event.

In [ ]:
# Solution: jet multiplicity histogram
# `ak.num(events.Jet)` counts how many jets each event has.
# This stays per-event (we do NOT flatten), so the histogram is "events vs Njets".
njets = ak.num(events.Jet)

plt.figure()
plt.hist(ak.to_numpy(njets), bins=15, range=(0, 15), edgecolor="black", alpha=0.7)
plt.xlabel("Jet multiplicity")
plt.ylabel("Events")
plt.title("Number of jets per event")
plt.show()

## Lepton content (Ex 1.5)

Inspect basic kinematics for electrons and muons using the loaded `events` object:
- Print the first few values of **pt, eta, and phi** for `events.Electron` and `events.Muon`.
- Count how many events have at least one electron or at least one muon.


In [ ]:
# Solution: lepton content
# In NanoAOD, electrons and muons are stored as collections (jagged arrays).
# Printing the first few entries helps verify the branch mapping and units.
print("Electron pt (first 5 events):", events.Electron.pt[:5])
print("Electron eta (first 5 events):", events.Electron.eta[:5])
print("Electron phi (first 5 events):", events.Electron.phi[:5])

print("Muon pt (first 5 events):", events.Muon.pt[:5])
print("Muon eta (first 5 events):", events.Muon.eta[:5])
print("Muon phi (first 5 events):", events.Muon.phi[:5])

# Count leptons per event (axis=1 = along the per-event jagged dimension)
# This gives per-event multiplicities that we can threshold.
n_ele = ak.count(events.Electron.pt, axis=1)
n_mu = ak.count(events.Muon.pt, axis=1)
print("Events with >=1 electron:", ak.sum(n_ele >= 1))
print("Events with >=1 muon:", ak.sum(n_mu >= 1))

## Lepton kinematic plots (Ex 1.6)

Make simple histograms of lepton kinematics:
- Plot pt for electrons and muons (e.g. 0–200 GeV),
- Plot eta for electrons and muons,
- Plot phi for electrons and muons.


In [ ]:
# Example solution: lepton kinematic histograms
# For kinematic distributions, we typically want one entry per lepton.
# Flatten the jagged collections (events × leptons) into a 1D array of leptons.
el_pt = ak.flatten(events.Electron.pt)
el_eta = ak.flatten(events.Electron.eta)
el_phi = ak.flatten(events.Electron.phi)

mu_pt = ak.flatten(events.Muon.pt)
mu_eta = ak.flatten(events.Muon.eta)
mu_phi = ak.flatten(events.Muon.phi)

# Electron pT
plt.figure()
plt.hist(ak.to_numpy(el_pt), bins=50, range=(0, 200), edgecolor="black", alpha=0.7)
plt.xlabel("Electron p$_T$ [GeV]")
plt.ylabel("Electrons")
plt.title("Electron transverse momentum")
plt.show()

# Muon pT
plt.figure()
plt.hist(ak.to_numpy(mu_pt), bins=50, range=(0, 200), edgecolor="black", alpha=0.7)
plt.xlabel("Muon p$_T$ [GeV]")
plt.ylabel("Muons")
plt.title("Muon transverse momentum")
plt.show()

# (Eta/phi plots would be analogous: use el_eta/mu_eta and el_phi/mu_phi.)